## Index

1. [DXSUM: Diagnostic Summary](#1.-DXSUM:-Diagnostic-Summary)
2. [PTDEMOG: Participant Demographics](#2.-PTDEMOG:-Participant-Demographics)
3. [UCBERKELEY_AMY_6MM: Amyloid PET](#3.-UCBERKELEY_AMY_6MM:-Amyloid-PET)
4. [UCSFFSX7: FreeSurfer volumetry (MRI)](#4.-UCSFFSX7:-FreeSurfer-volumetry-(MRI))
5. [UPENNBIOMK_ROCHE_ELECSYS: CSF biomarkers](#5.-UPENNBIOMK_ROCHE_ELECSYS:-CSF-biomarkers)
6. [Visit-code normalization](#6.-Visit-code-normalization)
7. [Join funnel: row counts before the merge](#7.-Join-funnel:-row-counts-before-the-merge)
8. [Final join](#8.-Final-join)
9. [Post-join sanity checks](#9.-Post-join-sanity-checks)

# Building the joint dataset

Structure of this notebook: for each raw table, we load it, look at what it contains, and select **only** the columns we need.

All selections are then merged into a single table (RID x visit).

In [216]:
import pandas as pd
import numpy as np
from scipy import stats

DATA_DIR = "../datasets"
datadict = pd.read_csv(f"{DATA_DIR}/DATADIC_12Dec2025.csv", low_memory=False)

In [217]:
datadict.head(5)

,PHASE,CRFNAME,TBLNAME,FLDNAME,TEXT,TYPE,LENGTH,DD_CRF_VERSION,CODE,UNITS,STATUS,CODE_CHANGES,MAPPING_NOTES
0,ADNI1,ADAS-Cognitive Behavior,ADAS,PTID,Participant ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ADNI1,ADAS-Cognitive Behavior,ADAS,RID,Participant roster ID,N,38 digits,NaN,NaN,NaN,NaN,NaN,NaN
2,ADNI1,ADAS-Cognitive Behavior,ADAS,VISCODE,Visit code,T,20 characters,NaN,NaN,NaN,NaN,NaN,NaN
3,ADNI1,ADAS-Cognitive Behavior,ADAS,EXAMDATE,Examination Date,D,10,NaN,NaN,NaN,Redacted,NaN,ADAS Item Level and Sub Scores Redacted
4,ADNI1,ADAS-Cognitive Behavior,ADAS,VISDATE,Assessment EXAMDATE when present; otherwise Re...,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Utils (make a .py in the end)

In [218]:
def describe_columns(df, table_name, datadic):
    """Print each column of `df` on its own line, next to its DATADIC description."""
    lookup = (
        datadic[datadic["TBLNAME"] == table_name]
        .drop_duplicates(subset="FLDNAME")
        .set_index("FLDNAME")["TEXT"]
    )
    width = max(len(c) for c in df.columns)
    print(f"{table_name}: {df.shape[0]} rows x {df.shape[1]} columns\n")
    for col in df.columns:
        desc = lookup.get(col, "(not found in DATADIC)")
        print(f"  {col:<{width}}  {desc}")
        
        
        
def profile_selected(df, cols, table_name, datadic, id_cols=None,
                      category_threshold=15, max_categories=8):
    """For each selected column: DATADIC meaning, missing%, and a value summary."""
    id_cols = set(id_cols or [])
    lookup = (
        datadic[datadic["TBLNAME"] == table_name]
        .drop_duplicates(subset="FLDNAME")
        .set_index("FLDNAME")["TEXT"]
    )
    print(f"{table_name}: selected columns ({len(cols)}):\n")
    for col in cols:
        if col not in df.columns:
            print(f"  {col}: NOT FOUND in dataframe\n")
            continue
        desc = lookup.get(col, "(not found in DATADIC)")
        s = df[col]
        n_missing = s.isna().sum()
        pct_missing = n_missing / len(s) * 100
        print(f"  {col}")
        print(f"    meaning:  {desc}")
        print(f"    missing:  {n_missing}/{len(s)} ({pct_missing:.1f}%)")

        if col in id_cols:
            print(f"    values:   identifier, range {s.min()}-{s.max()}, {s.nunique()} unique")
        elif "DATE" in col.upper() and s.dtype == object:
            s_dt = pd.to_datetime(s, errors="coerce")
            print(f"    values:   {s_dt.min().date()} to {s_dt.max().date()}")
        elif pd.api.types.is_numeric_dtype(s) and s.nunique() <= category_threshold:
            top = s.value_counts().sort_index()
            top_str = ", ".join(f"{k}={v}" for k, v in top.items())
            print(f"    values:   {top_str}  (coded categories)")
        elif pd.api.types.is_numeric_dtype(s):
            d = s.describe()
            print(f"    values:   min={d['min']:.2f}  mean={d['mean']:.2f}  "
                  f"median={d['50%']:.2f}  max={d['max']:.2f}")
        else:
            top = s.value_counts().head(max_categories)
            top_str = ", ".join(f"{k}={v}" for k, v in top.items())
            n_unique = s.nunique()
            suffix = "" if n_unique <= max_categories else f"  (+{n_unique - max_categories} more)"
            print(f"    values:   {top_str}{suffix}")
        print()

## 1. DXSUM: Diagnostic Summary

Contains the clinical diagnosis at every visit, plus many detail fields on subtypes (vascular dementia, Parkinson's, depression, diagnosis confidence...). 

We only need the macro stage (CN/MCI/Dementia), not the clinical sub-classification.

In [219]:
dxsum = pd.read_csv(f"{DATA_DIR}/DXSUM_12Dec2025.csv", low_memory=False)
describe_columns(dxsum, "DXSUM", datadict)

DXSUM: 15931 rows x 41 columns

  PHASE                 (not found in DATADIC)
  PTID                  Participant ID
  RID                   Participant roster ID
  VISCODE               Visit code
  VISCODE2              Translated visit code
  EXAMDATE              Date Form Completed
  DIAGNOSIS             1.  Which best describes the participant's current diagnosis?
  DXNORM                Normal
  DXNODEP               Mild Depression
  DXMCI                 Mild Cognitive Impairment
  DXMDES                If Mild Cognitive Impairment, select any that apply:
  DXMPTR1               1. Subjective memory complaint
  DXMPTR2               2. Informant memory complaint
  DXMPTR3               3. Normal general cognitive function
  DXMPTR4               4. Normal activities of daily living
  DXMPTR5               5. Objective memory impairment for age and education
  DXMPTR6               6. Not demented by diagnostic criteria
  DXMDUE                If MCI
  DXMOTHET              I

What we select:

- **`RID`**: subject identifier (required for every join)
- **`VISCODE2`**: visit code, used to align tables over time
- **`EXAMDATE`**: visit date, needed to compute age and to order visits in
  time (required for Model 2, time-to-conversion)
- **`DIAGNOSIS`**: the stage (1=CN, 2=MCI, 3=Dementia, per DATADIC): the
  reference target for all three models
- **`DXDSEV`**: dementia severity (1=Mild, 2=Moderate, 3=Severe, per DATADIC), a sub-level valid only within `DIAGNOSIS=3`.

The other values are more specific subtypes.

In [220]:
dxsum_selected = dxsum[["RID", "VISCODE2", "EXAMDATE", "DIAGNOSIS", "DXDSEV"]].copy()

# FIX: -4 = "not applicable" sentinel (question 3a not asked unless DIAGNOSIS=3), not a real severity value
dxsum_selected["DXDSEV"] = dxsum_selected["DXDSEV"].replace(-4, np.nan)

profile_selected(dxsum, ["RID", "VISCODE2", "EXAMDATE", "DIAGNOSIS"], "DXSUM", datadict, id_cols=["RID"])

DXSUM: selected columns (4):

  RID
    meaning:  Participant roster ID
    missing:  0/15931 (0.0%)
    values:   identifier, range 2-10898, 3788 unique

  VISCODE2
    meaning:  Translated visit code
    missing:  11/15931 (0.1%)
    values:   bl=3011, sc=2839, m12=1848, m06=1616, m24=1559, m36=944, m48=832, m72=456  (+34 more)

  EXAMDATE
    meaning:  Date Form Completed
    missing:  114/15931 (0.7%)
    values:   2005-09-29 to 2025-12-11

  DIAGNOSIS
    meaning:  1.  Which best describes the participant's current diagnosis?
    missing:  45/15931 (0.3%)
    values:   1.0=6304, 2.0=6580, 3.0=3002  (coded categories)



In [221]:
# Combined 5-level ordinal stage
# TODO: maybe it is a good idea to delete DIAGNOSIS and DXDSEV
stage_map = {1: "CN", 2: "MCI"}
sev_label = {1: "Mild", 2: "Moderate", 3: "Severe"}
def build_stage_full(row):
    if row["DIAGNOSIS"] in stage_map:
        return stage_map[row["DIAGNOSIS"]]
    if row["DIAGNOSIS"] == 3:
        s = sev_label.get(row["DXDSEV"])
        return f"Dementia-{s}" if s else "Dementia-Unspecified"
    return np.nan
dxsum_selected["STAGE_FULL"] = dxsum_selected.apply(build_stage_full, axis=1)

print("\nSTAGE_FULL distribution:")
print(dxsum_selected["STAGE_FULL"].value_counts())


STAGE_FULL distribution:
STAGE_FULL
MCI                     6580
CN                      6304
Dementia-Mild           2308
Dementia-Moderate        593
Dementia-Severe           85
Dementia-Unspecified      16
Name: count, dtype: int64


## 2. PTDEMOG: Participant Demographics

Beyond the core demographics (age, sex, education, marital status), the table has a large language block, origin/immigration fields, gender identity/orientation, work history, and symptom-onset dates.

In [222]:
ptdemog = pd.read_csv(f"{DATA_DIR}/PTDEMOG_12Dec2025.csv", low_memory=False)
describe_columns(ptdemog, "PTDEMOG", datadict)

PTDEMOG: 6222 rows x 84 columns

  PHASE                 (not found in DATADIC)
  PTID                  Participant ID
  RID                   Participant roster ID
  VISCODE               Visit code
  VISCODE2              Translated visit code
  VISDATE               Assessment EXAMDATE when present; otherwise Registry EXAMDATE
  PTSOURCE              Information Source
  PTGENDER              1. Participant Gender
  PTDOB                 2. Participant Date of Birth
  PTDOBYY               2b. Participant Year of Birth
  PTHAND                3. Participant Handedness
  PTMARRY               4. Participant Marital Status
  PTEDUCAT              5. Participant Education
  PTWORKHS              5a. Does the participant have a work history sufficient to exclude mental retardation? <!--Participant Education-->
  PTWORK                6a. Primary occupation during most of adult life
  PTNOTRT               7. Participant Retired?
  PTRTYR                Retirement Date
  PTHOME          

What we select:

- `PTGENDER` (1=Male, 2=Female): sex-related differences in AD risk and atrophy pattern are consistent in the literature.
- `PTDOB`: date of birth, used to compute precise age at each visit.
- `PTEDUCAT` (years, 0-30): proxy for cognitive reserve.
- `PTMARRY` (1=Married, 2=Widowed, 3=Divorced, 4=Never married, 5=Unknown, 6=Domestic partnership from ADNI4): a proxy for social isolation.
- `PTNOTRT` (retired y/n): occupational engagement is an independently consolidated cognitive-reserve factor.

*(`RID`, `VISCODE2` also kept, needed to key/align records.)*

Unfortunately many attributes on work status, language, and origin/immigration are theoretically relevant, but are either missing for most participants or redundant with the above.

In [223]:
df = ptdemog.copy()

# FIX:
# -4: standard ADNI "not applicable" sentinel (all columns)
# -1 on PTEDUCAT: not a documented DATADIC code (valid range is 0..20) isolated noise on 4/6222 rows, treated as missing
# 2 on PTNOTRT: DATADIC-documented "Not Applicable" for this field specifically

for col in ["PTGENDER", "PTEDUCAT", "PTMARRY", "PTNOTRT"]:
    df[col] = df[col].replace(-4, np.nan)
df["PTEDUCAT"] = df["PTEDUCAT"].replace(-1, np.nan)
df["PTNOTRT"] = df["PTNOTRT"].replace(2, np.nan)
df["PTDOB"] = df["PTDOB"].replace("", np.nan)

df = df.sort_values(["RID", "VISCODE2"])

fill_cols = ["PTGENDER", "PTDOB", "PTEDUCAT", "PTMARRY", "PTNOTRT"]
before = {col: df[col].isna().sum() for col in fill_cols}

for col in ["PTGENDER", "PTDOB", "PTEDUCAT"]:
    df[col] = df.groupby("RID")[col].transform(lambda s: s.ffill().bfill())
for col in ["PTMARRY", "PTNOTRT"]:
    df[col] = df.groupby("RID")[col].transform(lambda s: s.ffill())

after = {col: df[col].isna().sum() for col in fill_cols}

print(f"{'Column':<10} {'before':>8} {'after':>8} {'recovered':>10}")
for col in fill_cols:
    print(f"{col:<10} {before[col]:>8} {after[col]:>8} {before[col]-after[col]:>10}")

ptdemog_selected = df[["RID", "VISCODE2"] + fill_cols].copy()
print(f"\nSelected: {list(ptdemog_selected.columns)}")

/var/folders/n9/r8x6svc56fjb_2b7zb4tztr80000gn/T/ipykernel_77129/2142985557.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df.groupby("RID")[col].transform(lambda s: s.ffill().bfill())


Column       before    after  recovered
PTGENDER        186       10        176
PTDOB           188       12        176
PTEDUCAT        210       28        182
PTMARRY          35       35          0
PTNOTRT         125      122          3

Selected: ['RID', 'VISCODE2', 'PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTMARRY', 'PTNOTRT']


In [224]:
profile_selected(
    ptdemog_selected, fill_cols, "PTDEMOG", datadict, id_cols=[]
)

PTDEMOG: selected columns (5):

  PTGENDER
    meaning:  1. Participant Gender
    missing:  10/6222 (0.2%)
    values:   1.0=3099, 2.0=3113  (coded categories)

  PTDOB
    meaning:  2. Participant Date of Birth
    missing:  12/6222 (0.2%)
    values:   06/1944=35, 07/1943=32, 05/1935=32, 07/1947=29, 11/1932=28, 08/1935=27, 07/1940=27, 12/1926=27  (+611 more)

  PTEDUCAT
    meaning:  5. Participant Education
    missing:  28/6222 (0.5%)
    values:   min=3.00  mean=15.94  median=16.00  max=20.00

  PTMARRY
    meaning:  4. Participant Marital Status
    missing:  35/6222 (0.6%)
    values:   1.0=4382, 2.0=724, 3.0=676, 4.0=348, 5.0=27, 6.0=30  (coded categories)

  PTNOTRT
    meaning:  7. Participant Retired?
    missing:  122/6222 (2.0%)
    values:   0.0=1295, 1.0=4805  (coded categories)



In [225]:
na_mask = ptdemog_selected[fill_cols].isna()
all_missing = na_mask.all(axis=1)
any_missing = na_mask.any(axis=1)

print(f"Rows with >=1 missing feature: {any_missing.sum()}/{len(ptdemog_selected)} "
      f"({any_missing.mean()*100:.1f}%)")
print(f"Rows with ALL features missing (fully unrecoverable): "
      f"{all_missing.sum()}/{len(ptdemog_selected)} ({all_missing.mean()*100:.1f}%)")
print("\nDistribution of # missing features per row:")
print(na_mask.sum(axis=1).value_counts().sort_index())

Rows with >=1 missing feature: 132/6222 (2.1%)
Rows with ALL features missing (fully unrecoverable): 10/6222 (0.2%)

Distribution of # missing features per row:
0    6090
1      95
2      19
3       8
5      10
Name: count, dtype: int64


## 3. UCBERKELEY_AMY_6MM: Amyloid PET
Very huge table with 344 attributes but not independent: they break down into 10 identification/technical columns, 3 QC/tracer columns, 5 whole-brain summary measures, 10 reference-region measures (denominators used to normalize SUVR), and 158+158 = 316 regional measures.

In [226]:
amy = pd.read_csv(f"{DATA_DIR}/UCBERKELEY_AMY_6MM_12Dec2025.csv", low_memory=False)
describe_columns(amy, "UCBERKELEY_AMY_6MM", datadict)

UCBERKELEY_AMY_6MM: 4582 rows x 344 columns

  LONIUID                                 LONI Image ID
  PTID                                    ADNI Participant ID consisting of 3-digit "SITEID" and 4-digit "RID"
  RID                                     Participant roster ID (unique participant identifier that should be used to merge data)
  VISCODE                                 Visit code
  VISCODE2                                Translated visit code
  SCANDATE                                Scan acquisition date
  SITEID                                  Site ID
  PROCESSDATE                             Date quantification pipeline was run
  IMAGE_RESOLUTION                        Spatial resolution in mm^3 FWHM of Step4 Preprocessed PET image. See UCBERKELEY PET 8mm-to-6mm Transformation Methods PDF on LONI for details.
  qc_flag                                 Quality control flag based on visual inspection: 2=Pass;1=Partial pass; 0=Fail; -1=Not assessed; -2=Cannot be processed; 

We selected: 

- **`CENTILOIDS`**: standardized amyloid burden measure (the "A" in A/T/N), comparable across scanners/tracers by construction.
- **`AMYLOID_STATUS`**: binary positive/negative, useful as a sanity check and as a possible categorical alternative to CENTILOIDS.
- **`SUMMARY_SUVR`**: the raw measure underlying CENTILOIDS, it already aggregates nearly all the informative cortical-region signal.
- **`AMYLOID_STATUS_COMPOSITE_REF`**: a second binary reference.
- **`qc_flag`**: coded QC a technical reliability flag (2=Pass, 1=Partial pass, 0=Fail, -1=Not assessed, -2=Cannot be processed) telling us how much to trust `CENTILOIDS`/ `SUMMARY_SUVR` for a given row.

*(`RID`, `VISCODE2` also kept, needed to key/align records.)*

In [227]:
amy_selected = amy[
    ["RID", "VISCODE2", "CENTILOIDS", "AMYLOID_STATUS",
     "AMYLOID_STATUS_COMPOSITE_REF", "SUMMARY_SUVR", "qc_flag"]
].copy()

profile_selected(
    amy_selected, amy_selected.columns, "UCBERKELEY_AMY_6MM", datadict, id_cols=[]
)

UCBERKELEY_AMY_6MM: selected columns (7):

  RID
    meaning:  Participant roster ID (unique participant identifier that should be used to merge data)
    missing:  0/4582 (0.0%)
    values:   min=21.00  mean=5020.07  median=4835.00  max=10875.00

  VISCODE2
    meaning:  Translated visit code
    missing:  0/4582 (0.0%)
    values:   bl=1960, m24=773, m48=501, m72=214, m60=138, m84=117, m96=102, m78=87  (+28 more)

  CENTILOIDS
    meaning:  Summary cortical SUVR normalized by whole cerebellum and transformed to Centiloids. All SUVRs in this file are normalized to the whole cerebellum unless otherwise noted. See UCBERKELEY Amyloid Processing Methods PDF on LONI for details.
    missing:  14/4582 (0.3%)
    values:   min=-39.00  mean=33.88  median=13.00  max=346.00

  AMYLOID_STATUS
    meaning:  Amyloid positivity determined by applying thresholds to cortical summary SUVR normalized by whole cerebellum; See UCBERKELEY Amyloid Processing Methods PDF on LONI for details.
    missing:  8

In [228]:
# Some evidence checks

# Do the two positivity flags actually agree?
comparable = amy_selected.dropna(subset=["AMYLOID_STATUS", "AMYLOID_STATUS_COMPOSITE_REF"])

# Sanity check on qc_flag: rows that failed / couldn't be processed (0, -2)
# should NOT still carry a CENTILOIDS value. If they do, qc_flag can't be
# trusted as a simple filter for excluding unreliable rows downstream.
bad_qc_with_centiloids = amy_selected[
    amy_selected["qc_flag"].isin([0, -2]) & amy_selected["CENTILOIDS"].notna()
]

# Redundancy check on the ~316 regional SUVR columns we chose NOT to keep
suvr_cols = [c for c in amy.columns if c.endswith("_SUVR")
             and c not in ["SUMMARY_SUVR", "WHOLECEREBELLUM_SUVR", "COMPOSITE_REF_SUVR",
                            "CEREBELLUM_CORTEX_SUVR", "ERODED_SUBCORTICALWM_SUVR",
                            "BRAINSTEM_SUVR", "TRACER_SUVR_WARNING"]]
cortical_gm = [c for c in suvr_cols if c.startswith("CTX_")]
non_cortical = [c for c in suvr_cols if not c.startswith("CTX_")]

# Per-column Pearson r against SUMMARY_SUVR, pairwise-dropping NaNs.
# columns with fewer than 30 overlapping observations are skipped as unreliable.
def corr_with_summary(cols):
    return pd.Series([
        amy[[c, "SUMMARY_SUVR"]].dropna().corr().iloc[0, 1]
        for c in cols if amy[[c, "SUMMARY_SUVR"]].dropna().shape[0] >= 30
    ])
cort_corr = corr_with_summary(cortical_gm)
noncort_corr = corr_with_summary(non_cortical)

# Missingness on a representative regional column
# NOTE: All regional columns share essentially the same missingness pattern 
# (a scan either has or doesn't have a full regional breakdown), so the first 
# column in the list stands in for the whole block.
base_missing = amy[suvr_cols[0]].isna()

pd.Series({
    "status_agreement_pct":        round((comparable["AMYLOID_STATUS"] == comparable["AMYLOID_STATUS_COMPOSITE_REF"]).mean() * 100, 1),
    "bad_qc_with_centiloids":      len(bad_qc_with_centiloids),
    "cortical_median_r":           round(cort_corr.median(), 3),
    "cortical_redundant_frac":     f"{(cort_corr>0.9).sum()}/{len(cort_corr)}",
    "noncortical_median_r":        round(noncort_corr.median(), 3),
    "entorhinal_r":                round(amy[["CTX_ENTORHINAL_SUVR", "SUMMARY_SUVR"]].dropna().corr().iloc[0,1], 3),
    "regional_missing_pct":        round(base_missing.mean() * 100, 1),
})

status_agreement_pct         93.5
bad_qc_with_centiloids          0
cortical_median_r           0.946
cortical_redundant_frac    78/102
noncortical_median_r        0.233
entorhinal_r                0.727
regional_missing_pct          7.7
dtype: object

- **Status agreement (93.5%)**:<br> The two positivity flags mostly agree, the ~6.5% disagreement likely reflects borderline cases near the positivity cutoff.
- **QC flag reliability (0 bad rows)**:<br> No row with a failed/unprocessable `qc_flag` (0, -2) still carries a `CENTILOIDS` value so the flag can be trusted as a simple filter downstream, no extra cross-check needed.
- **Cortical redundancy (median r = 0.946, 78/102 regions with r > 0.9)**:<br> Strongly confirms dropping the ~316 regional AMY columns in favor of `SUMMARY_SUVR`, most cortical regional detail is redundant with the summary measure already kept.
- **Non-cortical regions (median r = 0.233)**:<br> Low correlation with cortical amyloid signal, as expected biologically (basal ganglia, white matter, ventricles aren't informative for amyloid).
- **Entorhinal cortex (r = 0.727)**:<br> Notably lower than the cortical median despite being a cortical region. Worth keeping in mind if regional AMY is revisited later.

## 4. UCSFFSX7: FreeSurfer volumetry (MRI)

The other huge table with 347 columns, almost all volumes/areas/thickness for individual brain structures from the standard FreeSurfer `aparc`/`aseg` parcellation (pattern `ST<region>{SV,CV,SA,TA,TS}`: Subcortical Volume, Cortical Volume, Surface Area, Thickness Average, Thickness Std). 
We need the hippocampus, the total intracranial volume to normalize for head size, and the entorhinal cortex and amygdala.

In [229]:
fsx7 = pd.read_csv(f"{DATA_DIR}/UCSFFSX7_12Dec2025.csv", low_memory=False)
describe_columns(fsx7, "UCSFFSX7", datadict)

UCSFFSX7: 12151 rows x 347 columns

  PHASE           Phase of the study in which this observation was collected
  PTID            Participant Identifier for the in-clinic ADNI study
  RID             Participant Identifier for the in-clinic ADNI study
  VISCODE         VISCODE of the Image
  VISCODE2        (not found in DATADIC)
  IMAGEUID        Image UID of image that was processed.
  FIELD_STRENGTH  the field strength (tesla) of the image
  EXAMDATE        When the image was collected.
  RUNDATE         When the image was processed.
  STATUS          partial = No QC, not final. Complete = QC and finalized.
  FSVER           Specific version of FreeSurfer used. All FS 7.x results are compatible.
  OVERALLQC       An overall quality rating. Refer to additional QC variables for Partial rating.
  TEMPQC          QC rating of temporal lobe. Fail affects the following regions: LeftTemporalPole (ST60); RightTemporalPole (ST119); LeftFusiform (ST26); RightFusiform (ST85); LeftSuperiorTemp

We select:

- `ST29SV`: left hippocampal volume 
- `ST88SV`: right hippocampal volume 
- `ST10CV`: total intracranial volume/eTIV, used to normalize hippocampal volumes for head size
- `ST24CV` / `ST83CV`: left/right **entorhinal cortex** volume. The entorhinal cortex is documented as one of the earliest sites of AD pathology, showing atrophy *before* the hippocampus, and is specifically reported as the more sensitive measure for distinguishing controls from early MCI, while the hippocampus remains better for MCI-vs-AD. Relevant especially for Model 1 (anomaly detection on CNan earlier-marking region may improve sensitivity to early deviation) and Model 2 (conversion risk, particularly the CN→MCI transition)
- `ST12SV` / `ST71SV`: left/right **amygdala** volume. Not part of
  the original A/T/N brief, added after the analytical check in
  section 8 below showed an association with `DIAGNOSIS` nearly as
  strong as the hippocampus; this is consistent with literature
  describing early amygdala involvement on tau-PET and studies
  examining amygdala/entorhinal atrophy jointly across the AD
  spectrum
- `OVERALLQC`: overall segmentation QC flag; kept and checked below
  as a candidate filter, since a near-zero entorhinal volume found
  in the evidence check (not QC-driven selection, just a plausibility
  scan) suggests QC can't be ignored the way it was tentatively
  treated before

**Considered but not included**: `HIPPOQC` / `VENTQC` — per DATADIC,
`HIPPOQC` actually covers fusiform/entorhinal/parahippocampal/amygdala
QC, *not* the hippocampus itself (a misleading name worth noting, not
taking at face value); `VENTQC` covers ventricle-related regions. Since
we don't currently model ventricles and the hippocampus has its own
general QC coverage, we don't add these as separate columns, but they
remain available in the raw table if a future model calls for them.
`ST40CV` (left middle temporal cortical volume) is also worth flagging
here: it ranks in the top 10 by association with `DIAGNOSIS` in the
descriptive scan (section 8), above right entorhinal — we don't add
it to keep the feature set anchored to the A/T/N-motivated regions
already selected (hippocampus, entorhinal, amygdala), but it's a
candidate if a future iteration widens the temporal-lobe features.

In [230]:
fsx7_selected = fsx7[
    ["RID", "VISCODE2", "ST29SV", "ST88SV", "ST10CV",
     "ST24CV", "ST83CV", "ST12SV", "ST71SV", "OVERALLQC"]
].rename(columns={
    "ST29SV": "HIPPO_L", "ST88SV": "HIPPO_R", "ST10CV": "ETIV",
    "ST24CV": "ENTORHINAL_L", "ST83CV": "ENTORHINAL_R",
    "ST12SV": "AMYGDALA_L", "ST71SV": "AMYGDALA_R",
}).copy()

# We treat sub-100mm^3 volumes on these specific regions as failed-segmentation 
# noise, not a real value (100mm^3 is far below the smallest plausible
# entorhinal/amygdala/hippocampus volume in this cohort's own distribution)
vol_cols = ["HIPPO_L", "HIPPO_R", "ENTORHINAL_L", "ENTORHINAL_R",
            "AMYGDALA_L", "AMYGDALA_R", "ETIV"]
implausible_floor = 100  # mm^3

before_floor = {c: (fsx7_selected[c] < implausible_floor).sum() for c in vol_cols}
for c in vol_cols:
    fsx7_selected.loc[fsx7_selected[c] < implausible_floor, c] = np.nan

print(f"{'Column':<14} {'flagged as implausible (<100mm^3)':>34}")
for c in vol_cols:
    print(f"{c:<14} {before_floor[c]:>34}")

fsx7_selected["HIPPO_TOTAL"] = fsx7_selected["HIPPO_L"] + fsx7_selected["HIPPO_R"]
fsx7_selected["ENTORHINAL_TOTAL"] = fsx7_selected["ENTORHINAL_L"] + fsx7_selected["ENTORHINAL_R"]
fsx7_selected["AMYGDALA_TOTAL"] = fsx7_selected["AMYGDALA_L"] + fsx7_selected["AMYGDALA_R"]

# FIX: We switch to residualizing each volume against ETIV via linear regression,
# and extend the same correction to entorhinal/amygdala (raw correlations
# with ETIV were comparable to or higher than the hippocampus').
from sklearn.linear_model import LinearRegression

def etiv_residual(df, vol_col, etiv_col="ETIV"):
    """Head-size-corrected volume: residual of vol_col ~ ETIV (linear fit)."""
    mask = df[[vol_col, etiv_col]].notna().all(axis=1)
    reg = LinearRegression().fit(df.loc[mask, [etiv_col]], df.loc[mask, vol_col])
    resid = pd.Series(np.nan, index=df.index)
    resid.loc[mask] = df.loc[mask, vol_col] - reg.predict(df.loc[mask, [etiv_col]])
    return resid

fsx7_selected["HIPPO_NORM"] = etiv_residual(fsx7_selected, "HIPPO_TOTAL")
fsx7_selected["ENTORHINAL_NORM"] = etiv_residual(fsx7_selected, "ENTORHINAL_TOTAL")
fsx7_selected["AMYGDALA_NORM"] = etiv_residual(fsx7_selected, "AMYGDALA_TOTAL")

print(f"\nSelected/derived: {list(fsx7_selected.columns)}")

Column          flagged as implausible (<100mm^3)
HIPPO_L                                         0
HIPPO_R                                         0
ENTORHINAL_L                                    2
ENTORHINAL_R                                    0
AMYGDALA_L                                      0
AMYGDALA_R                                      0
ETIV                                            0

Selected/derived: ['RID', 'VISCODE2', 'HIPPO_L', 'HIPPO_R', 'ETIV', 'ENTORHINAL_L', 'ENTORHINAL_R', 'AMYGDALA_L', 'AMYGDALA_R', 'OVERALLQC', 'HIPPO_TOTAL', 'ENTORHINAL_TOTAL', 'AMYGDALA_TOTAL', 'HIPPO_NORM', 'ENTORHINAL_NORM', 'AMYGDALA_NORM']


In [231]:
profile_selected(
    fsx7_selected,
    fsx7_selected.columns,
    "UCSFFSX7", datadict, id_cols=[]
)

UCSFFSX7: selected columns (16):

  RID
    meaning:  Participant Identifier for the in-clinic ADNI study
    missing:  0/12151 (0.0%)
    values:   min=2.00  mean=3345.91  median=4020.00  max=10898.00

  VISCODE2
    meaning:  (not found in DATADIC)
    missing:  3/12151 (0.0%)
    values:   sc=2243, m12=1732, m06=1516, m24=1366, scmri=940, m03=763, m36=556, m48=468  (+34 more)

  HIPPO_L
    meaning:  (not found in DATADIC)
    missing:  145/12151 (1.2%)
    values:   min=1188.70  mean=3403.16  median=3437.35  max=5684.60

  HIPPO_R
    meaning:  (not found in DATADIC)
    missing:  145/12151 (1.2%)
    values:   min=1047.60  mean=3500.23  median=3548.85  max=5567.90

  ETIV
    meaning:  (not found in DATADIC)
    missing:  5/12151 (0.0%)
    values:   min=427249.65  mean=1521395.30  median=1485603.35  max=5158451.28

  ENTORHINAL_L
    meaning:  (not found in DATADIC)
    missing:  163/12151 (1.3%)
    values:   min=263.00  mean=1726.49  median=1723.00  max=4404.00

  ENTORHINAL_R


In [232]:
# Evidence checks

status_qc = fsx7.assign(has_qc=fsx7["OVERALLQC"].notna())
status_vs_qc = pd.crosstab(status_qc["STATUS"], status_qc["has_qc"])

fail_with_hippo = fsx7_selected[
    (fsx7_selected["OVERALLQC"] == "Fail") & fsx7_selected["HIPPO_NORM"].notna()
]

r_hippo_total_etiv = fsx7_selected[["HIPPO_TOTAL", "ETIV"]].dropna().corr().iloc[0, 1]
r_hippo_norm_etiv = fsx7_selected[["HIPPO_NORM", "ETIV"]].dropna().corr().iloc[0, 1]
r_entorhinal_total_etiv = fsx7_selected[["ENTORHINAL_TOTAL", "ETIV"]].dropna().corr().iloc[0, 1]
r_entorhinal_norm_etiv = fsx7_selected[["ENTORHINAL_NORM", "ETIV"]].dropna().corr().iloc[0, 1]
r_amygdala_total_etiv = fsx7_selected[["AMYGDALA_TOTAL", "ETIV"]].dropna().corr().iloc[0, 1]
r_amygdala_norm_etiv = fsx7_selected[["AMYGDALA_NORM", "ETIV"]].dropna().corr().iloc[0, 1]

print(status_vs_qc)
pd.Series({
    "rows_with_qc_pct":            round(status_qc["has_qc"].mean() * 100, 1),
    "fail_qc_with_hippo_norm":     len(fail_with_hippo),
    "hippo_total_etiv_r":          round(r_hippo_total_etiv, 3),
    "hippo_norm_etiv_r":           round(r_hippo_norm_etiv, 3),
    "entorhinal_total_etiv_r":     round(r_entorhinal_total_etiv, 3),
    "entorhinal_norm_etiv_r":      round(r_entorhinal_norm_etiv, 3),
    "amygdala_total_etiv_r":       round(r_amygdala_total_etiv, 3),
    "amygdala_norm_etiv_r":        round(r_amygdala_norm_etiv, 3),
})

has_qc    False  True 
STATUS                
complete      0   1065
partial   11086      0


rows_with_qc_pct           8.800
fail_qc_with_hippo_norm    0.000
hippo_total_etiv_r         0.117
hippo_norm_etiv_r          0.000
entorhinal_total_etiv_r    0.145
entorhinal_norm_etiv_r    -0.000
amygdala_total_etiv_r      0.102
amygdala_norm_etiv_r       0.000
dtype: float64

- **`STATUS` vs `OVERALLQC` coverage: confirmed.** The 91.2% missing rate matches `STATUS="partial"` exactly: `OVERALLQC` isn't "unknown quality," it's "run not yet finalized" (8.8% of rows have `STATUS="complete"`, and these are exactly the rows with a populated QC value).
- **QC reliability: confirmed, reliable filter.** No row with `OVERALLQC="Fail"` carries a populated `HIPPO_NORM` value (0 cases), the same behavior as `qc_flag` in AMY.
- **`ETIV` normalization: confirmed.** `hippo_norm_etiv_r`, `entorhinal_norm_etiv_r`, and `amygdala_norm_etiv_r` are all 0.000, vs raw correlations of 0.117, 0.145, and 0.102. The residual method removes the head-size confound completely on all three regions, unlike the earlier ratio (which gave r=-0.530 instead of removing it).

## 5. UPENNBIOMK_ROCHE_ELECSYS: CSF biomarkers

The table is already focused. We need the CSF counterparts to the imaging biomarkers above: beta-amyloid (a second "A" measure, independent of PET) and tau (the "T" in A/T/N).

In [233]:
csf = pd.read_csv(f"{DATA_DIR}/UPENNBIOMK_ROCHE_ELECSYS_12Dec2025.csv", low_memory=False)
describe_columns(csf, "UPENNBIOMK_ROCHE_ELECSYS", datadict)

UPENNBIOMK_ROCHE_ELECSYS: 3174 rows x 13 columns

  PHASE         ADNI phase
  PTID          Participant ID
  RID           Participant roster ID
  VISCODE2      Months from baseline rounded to nearest 6 months
  EXAMDATE      Date sample collected
  BATCH         Sample batch
  RUNDATE       Date of experiment
  ABETA40       ABETA40 result
  ABETA42       ABETA42 result
  TAU           TAU result
  PTAU          PTAU result
  COMMENT       Supplementary notes for missing values and outlier
  update_stamp  (not found in DATADIC)


What we select:

- **`ABETA42`**, **`ABETA40`**: CSF beta-amyloid. Their ratio is a second "A" indicator, independent of PET, but `ABETA40` isn't missing at random: it's essentially only available from ADNI3 onward (see evidence check below).
- **`TAU`**, **`PTAU`**: total and phosphorylated tau. `PTAU` is the "T" in A/T/N (specific to AD-type tau pathology). `TAU` reflects neurodegeneration more generally.
- **`COMMENT`**: not a free-text note, it flags when a value falls outside the instrument's calibrated range.

*(`RID`, `VISCODE2` also kept, needed to key/align records.)*

Derived: **`ABETA_RATIO`** (`ABETA42 / ABETA40`).

In [234]:
csf_selected = csf[["RID", "VISCODE2", "ABETA42", "ABETA40", "TAU", "PTAU", "COMMENT"]].copy()

csf_selected["ABETA_RATIO"] = pd.to_numeric(csf_selected["ABETA42"], errors="coerce") / \
    pd.to_numeric(csf_selected["ABETA40"], errors="coerce")

# FIX: COMMENT flags values outside the instrument's calibrated range. For TAU and PTAU
# this always means value=NaN.
csf_selected["ABETA42_CENSORED"] = (
    csf_selected["COMMENT"].str.contains("abeta42", case=False, na=False)
    & csf_selected["ABETA42"].notna()
)

In [235]:
profile_selected(
    csf_selected, csf_selected.columns, "UPENNBIOMK_ROCHE_ELECSYS", datadict, id_cols=[]
)

UPENNBIOMK_ROCHE_ELECSYS: selected columns (9):

  RID
    meaning:  Participant roster ID
    missing:  0/3174 (0.0%)
    values:   min=3.00  mean=3325.72  median=4190.50  max=7073.00

  VISCODE2
    meaning:  Months from baseline rounded to nearest 6 months
    missing:  0/3174 (0.0%)
    values:   bl=1621, m24=541, m12=324, m48=234, m36=91, m72=64, m60=53, m96=34  (+20 more)

  ABETA42
    meaning:  ABETA42 result
    missing:  7/3174 (0.2%)
    values:   min=203.00  mean=1062.01  median=870.40  max=4779.00

  ABETA40
    meaning:  ABETA40 result
    missing:  2240/3174 (70.6%)
    values:   min=841.00  mean=18130.83  median=17700.00  max=37480.00

  TAU
    meaning:  TAU result
    missing:  15/3174 (0.5%)
    values:   min=80.08  mean=286.72  median=256.90  max=1018.00

  PTAU
    meaning:  PTAU result
    missing:  27/3174 (0.9%)
    values:   min=8.00  mean=27.27  median=23.54  max=108.50

  COMMENT
    meaning:  Supplementary notes for missing values and outlier
    missing:  2

In [236]:
# Evidence checks

# ABETA40 isn't missing at random: coverage is gated almost entirely by ADNI phase
abeta40_by_phase = pd.DataFrame({
    "n": csf["PHASE"].value_counts(),
    "pct_abeta40_present": csf.groupby("PHASE")["ABETA40"].apply(lambda s: s.notna().mean()).round(3),
})

# Do TAU and PTAU actually carry different signal, or is one redundant with the other?
r_tau_ptau = csf_selected[["TAU", "PTAU"]].dropna().corr().iloc[0, 1]

# Does ABETA_RATIO add signal beyond ABETA42 alone?
r_abeta42_ratio = csf_selected[["ABETA42", "ABETA_RATIO"]].dropna().corr().iloc[0, 1]

# ABETA_RATIO should be missing exactly when ABETA42 or ABETA40 is missing, no odd cases
abeta42_num = pd.to_numeric(csf_selected["ABETA42"], errors="coerce")
abeta40_num = pd.to_numeric(csf_selected["ABETA40"], errors="coerce")
ratio_missing_mismatch = (
    csf_selected["ABETA_RATIO"].isna() != (abeta42_num.isna() | abeta40_num.isna())
).sum()
implausible_ratio = csf_selected[csf_selected["ABETA_RATIO"] < 0]

# Does ABETA42_CENSORED actually do what it should: all values above 1700?
censored_vals = csf_selected.loc[csf_selected["ABETA42_CENSORED"], "ABETA42"]

print("ABETA40 presence by PHASE:")
print(abeta40_by_phase)

pd.Series({
    "abeta40_present_ADNI3_pct":         round(csf.loc[csf["PHASE"]=="ADNI3","ABETA40"].notna().mean()*100, 1),
    "abeta40_present_other_phases_pct":  round(csf.loc[csf["PHASE"]!="ADNI3","ABETA40"].notna().mean()*100, 1),
    "tau_ptau_r":                        round(r_tau_ptau, 3),
    "abeta42_ratio_r":                   round(r_abeta42_ratio, 3),
    "ratio_missing_mismatch_n":          ratio_missing_mismatch,
    "implausible_ratio_n":               len(implausible_ratio),
    "censored_all_above_1700":           bool((censored_vals > 1700).all()),
    "censored_value_range":              f"{censored_vals.min():.0f}-{censored_vals.max():.0f}",
})

ABETA40 presence by PHASE:
           n  pct_abeta40_present
PHASE                            
ADNI1    938                0.035
ADNI2   1295                0.091
ADNI3    769                0.999
ADNIGO   172                0.087


abeta40_present_ADNI3_pct                99.9
abeta40_present_other_phases_pct          6.9
tau_ptau_r                              0.978
abeta42_ratio_r                         0.793
ratio_missing_mismatch_n                    0
implausible_ratio_n                         0
censored_all_above_1700                  True
censored_value_range                1702-4779
dtype: object

- **`ABETA40` is not missing at random: it's almost exclusively ADNI3 (99.9% vs 6.9% in other phases).** `ABETA_RATIO` inherits the same missingness structure (70.7%), so it needs to be accounted for separately from `ABETA42` when assessing the final dataset's coverage.
- **`TAU` and `PTAU` are not redundant, despite the high correlation.** The very high correlation (r = 0.978) confirms they capture essentially the same signal. keeping both is more an interpretability choice (generic neurodegeneration vs AD-specific tau pathology) than a source of extra information.
- **`ABETA_RATIO` adds signal beyond `ABETA42` alone (r = 0.793, not 1.0).** It isn't a redundant transformation of `ABETA42`, which justifies keeping it as a separate feature wherever `ABETA40` is available.
- **`ABETA_RATIO` consistency: confirmed.** No row (`ratio_missing_mismatch_n = 0`) where `ABETA_RATIO`'s missingness fails to match `ABETA42`/`ABETA40`'s, and no implausible ratios (`implausible_ratio_n = 0`, no negative values).
- **`ABETA42_CENSORED` does exactly what it should.** Every flagged censored value falls above the 1700 threshold (`censored_all_above_1700 = True`, range 1702-4779), consistent with `COMMENT` marking "Abeta42>1700" as instrument-saturation censoring rather than a real outlier to correct.

## 6. Visit-code normalization

The tables don't share the same baseline code: `PTDEMOG` and `FSX7` use `sc`/`scmri` (screening), `DXSUM` mixes `bl` and `sc`, `AMY` and `CSF` already use `bl`. 

These aren't different timepoints, they're the same baseline visit coded differently depending on ADNI phase/CRF. We normalize everything to `bl` before the join, otherwise baseline rows wouldn't align across tables and we'd silently lose most of the baseline population.

In [237]:
def normalize_viscode2(df, col="VISCODE2", out_col="VISCODE2_norm"):
    out = df.copy()
    out[out_col] = out[col].replace({"sc": "bl", "scmri": "bl"})
    return out

dxsum_selected = normalize_viscode2(dxsum_selected)
ptdemog_selected = normalize_viscode2(ptdemog_selected)
fsx7_selected = normalize_viscode2(fsx7_selected)
amy_selected = normalize_viscode2(amy_selected)
csf_selected = normalize_viscode2(csf_selected)

print("VISCODE2_norm value counts at baseline, per table:")
for name, df in [("DXSUM", dxsum_selected), ("PTDEMOG", ptdemog_selected),
                  ("FSX7", fsx7_selected), ("AMY", amy_selected), ("CSF", csf_selected)]:
    n_bl = (df["VISCODE2_norm"] == "bl").sum()
    print(f"  {name:10s} bl rows: {n_bl}")

VISCODE2_norm value counts at baseline, per table:
  DXSUM      bl rows: 5850
  PTDEMOG    bl rows: 4600
  FSX7       bl rows: 3295
  AMY        bl rows: 1960
  CSF        bl rows: 1621


## 7. Join funnel: row counts before the merge

Before merging, we check what we're actually about to join: how many rows each satellite table contributes, how many collapse when we deduplicate on `RID` x `VISCODE2_norm`, and how many rows actually find a match in the DXSUM backbone.

In [238]:
def join_funnel(df, table_name, key=["RID", "VISCODE2_norm"], backbone_keys=None):
    """Report row counts, duplicate collapse, and backbone match rate."""
    n_raw = len(df)
    n_dup_dropped = df.duplicated(subset=key, keep="first").sum()
    df_dedup = df.drop_duplicates(subset=key, keep="first")
    print(f"{table_name}:")
    print(f"  raw rows:                 {n_raw}")
    print(f"  duplicate RID x visit:    {n_dup_dropped}  (dropped via keep='first')")
    print(f"  rows after dedup:         {len(df_dedup)}")
    if backbone_keys is not None:
        matched = df_dedup.merge(backbone_keys, on=key, how="inner")
        n_matched = len(matched)
        print(f"  rows matching DXSUM backbone: {n_matched}/{len(df_dedup)} "
              f"({n_matched/len(df_dedup)*100:.1f}%)")
    print()
    return df_dedup

# Backbone reference keys (RID x visit with a known DIAGNOSIS)
dx_base_keys = (
    dxsum_selected.dropna(subset=["DIAGNOSIS"])
    [["RID", "VISCODE2_norm"]]
    .drop_duplicates()
)
print(f"DXSUM backbone: {len(dxsum_selected)} raw rows, "
      f"{len(dx_base_keys)} unique RID x visit with non-null DIAGNOSIS\n")

amy_dedup = join_funnel(amy_selected, "AMY", backbone_keys=dx_base_keys)
fsx7_dedup = join_funnel(fsx7_selected, "FSX7", backbone_keys=dx_base_keys)
csf_dedup = join_funnel(csf_selected, "CSF", backbone_keys=dx_base_keys)

# PTDEMOG is baseline-only, joined on RID alone — different key, checked separately
n_demo_dup = ptdemog_selected[ptdemog_selected["VISCODE2_norm"] == "bl"].duplicated(subset=["RID"], keep="first").sum()
print(f"PTDEMOG (baseline only):")
print(f"  raw bl rows:              {(ptdemog_selected['VISCODE2_norm'] == 'bl').sum()}")
print(f"  duplicate RID at bl:      {n_demo_dup}  (dropped via keep='first')")

DXSUM backbone: 15931 raw rows, 13827 unique RID x visit with non-null DIAGNOSIS

AMY:
  raw rows:                 4582
  duplicate RID x visit:    6  (dropped via keep='first')
  rows after dedup:         4576
  rows matching DXSUM backbone: 4549/4576 (99.4%)

FSX7:
  raw rows:                 12151
  duplicate RID x visit:    849  (dropped via keep='first')
  rows after dedup:         11302
  rows matching DXSUM backbone: 10477/11302 (92.7%)

CSF:
  raw rows:                 3174
  duplicate RID x visit:    0  (dropped via keep='first')
  rows after dedup:         3174


  rows matching DXSUM backbone: 3160/3174 (99.6%)

PTDEMOG (baseline only):
  raw bl rows:              4600
  duplicate RID at bl:      219  (dropped via keep='first')


## 8. Final join

DXSUM is the backbone: one row = one visit with a known diagnosis, since `DIAGNOSIS` is the target for all three models. Demographics are time-invariant (or close to it) and taken at baseline only; the other tables attach by `RID` x `VISCODE2_norm`.

In [239]:
dx_base = (
    dxsum_selected.dropna(subset=["DIAGNOSIS"])
    .rename(columns={"EXAMDATE": "EXAMDATE_DX"})
    .drop_duplicates(subset=["RID", "VISCODE2_norm"], keep="first")
)
demo_bl = (
    ptdemog_selected[ptdemog_selected["VISCODE2_norm"] == "bl"]
    .drop(columns=["VISCODE2", "VISCODE2_norm"])
    .drop_duplicates(subset=["RID"], keep="first")
)

joint = (
    dx_base.drop(columns=["VISCODE2"])
    .merge(demo_bl, on="RID", how="left")
    .merge(amy_dedup.drop(columns=["VISCODE2"]), on=["RID", "VISCODE2_norm"], how="left")
    .merge(fsx7_dedup.drop(columns=["VISCODE2"]), on=["RID", "VISCODE2_norm"], how="left")
    .merge(csf_dedup.drop(columns=["VISCODE2"]), on=["RID", "VISCODE2_norm"], how="left")
)

joint["PTDOB"] = pd.to_datetime(joint["PTDOB"], format="%m/%Y", errors="coerce")
joint["EXAMDATE_DX"] = pd.to_datetime(joint["EXAMDATE_DX"], errors="coerce")
joint["AGE"] = (joint["EXAMDATE_DX"] - joint["PTDOB"]).dt.days / 365.25

print(f"joint: {joint.shape[0]} rows x {joint.shape[1]} columns")
print(f"Columns: {list(joint.columns)}")
print(f"Unique subjects: {joint['RID'].nunique()}")

atn_complete = joint.dropna(subset=["CENTILOIDS", "PTAU", "HIPPO_NORM", "AGE"])
print(f"\nA/T/N-complete population: {atn_complete.shape[0]} rows, "
      f"{atn_complete['RID'].nunique()} subjects")

joint: 13827 rows x 38 columns
Columns: ['RID', 'EXAMDATE_DX', 'DIAGNOSIS', 'DXDSEV', 'STAGE_FULL', 'VISCODE2_norm', 'PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTMARRY', 'PTNOTRT', 'CENTILOIDS', 'AMYLOID_STATUS', 'AMYLOID_STATUS_COMPOSITE_REF', 'SUMMARY_SUVR', 'qc_flag', 'HIPPO_L', 'HIPPO_R', 'ETIV', 'ENTORHINAL_L', 'ENTORHINAL_R', 'AMYGDALA_L', 'AMYGDALA_R', 'OVERALLQC', 'HIPPO_TOTAL', 'ENTORHINAL_TOTAL', 'AMYGDALA_TOTAL', 'HIPPO_NORM', 'ENTORHINAL_NORM', 'AMYGDALA_NORM', 'ABETA42', 'ABETA40', 'TAU', 'PTAU', 'COMMENT', 'ABETA_RATIO', 'ABETA42_CENSORED', 'AGE']
Unique subjects: 3777

A/T/N-complete population: 1830 rows, 1241 subjects


## 9. Post-join sanity checks

Does `joint` behave the way it should, independent of any modeling decision? 

These aren't feature-selection checks (that's section 10), they're "did the join do what it was supposed to do".

In [240]:
# 1. No duplicate RID x visit in the backbone (would silently multiply rows on merge)
n_dup_backbone = joint.duplicated(subset=["RID", "VISCODE2_norm"]).sum()
print(f"Duplicate RID x visit in joint: {n_dup_backbone}")

# 2. Every row must have a non-null DIAGNOSIS (dx_base was filtered on this,
#    a left-join shouldn't be able to break it, but worth confirming)
n_missing_dx = joint["DIAGNOSIS"].isna().sum()
print(f"Rows with missing DIAGNOSIS after join: {n_missing_dx} (should be 0)")

# 3. AGE plausibility: catch sign errors or bad date parsing
age_stats = joint["AGE"].describe()
n_age_missing = joint["AGE"].isna().sum()
n_age_implausible = ((joint["AGE"] < 18) | (joint["AGE"] > 110)).sum()
print(f"\nAGE: min={age_stats['min']:.1f}  mean={age_stats['mean']:.1f}  "
      f"max={age_stats['max']:.1f}")
print(f"AGE missing: {n_age_missing}/{len(joint)} ({n_age_missing/len(joint)*100:.1f}%)")
print(f"AGE outside [18, 110] (implausible): {n_age_implausible}")

# 4. Subjects present in DXSUM but missing baseline demographics
#    (DXSUM has a visit, but PTDEMOG has no bl record for that RID)
rid_no_demo = joint[joint["PTGENDER"].isna()]["RID"].nunique()
print(f"\nSubjects with DXSUM visit(s) but no baseline PTDEMOG match: {rid_no_demo}")

# 5. Per-table match rate within joint (how much of joint is actually
#    populated per satellite table, vs structurally left-null)
for col, label in [("CENTILOIDS", "AMY"), ("HIPPO_NORM", "FSX7"), ("PTAU", "CSF")]:
    pct = joint[col].notna().mean() * 100
    print(f"{label:6s} coverage in joint ({col}): {pct:.1f}%")

Duplicate RID x visit in joint: 0
Rows with missing DIAGNOSIS after join: 0 (should be 0)



AGE: min=50.5  mean=75.3  max=99.1
AGE missing: 61/13827 (0.4%)
AGE outside [18, 110] (implausible): 0

Subjects with DXSUM visit(s) but no baseline PTDEMOG match: 3
AMY    coverage in joint (CENTILOIDS): 32.8%
FSX7   coverage in joint (HIPPO_NORM): 74.7%
CSF    coverage in joint (PTAU): 22.7%


In [241]:
from data_profiling import ProfileReport
profile = ProfileReport(joint, title="joint_dataset_report", minimal=False)
profile.to_file("reports/report_joint.html")

/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Export report to file: 100%|██████████| 1/1 [00:00<00:00,  7.10it/s]


In [242]:
import os

output_path = "../datasets/final.csv"
joint.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"  rows:    {joint.shape[0]}")
print(f"  columns: {joint.shape[1]}")
print(f"  size:    {os.path.getsize(output_path) / 1024:.1f} KB")

Saved: ../datasets/final.csv
  rows:    13827
  columns: 38
  size:    2820.7 KB
